<a href="https://colab.research.google.com/github/run-llama/llama_index/blob/main/docs/examples/vector_stores/ActianVectorAIVectorStoreDemo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Actian VectorAI Vector Store Demo

> This notebook shows how to use Actian VectorAI as a LlamaIndex vector store for ingestion and retrieval.

## Prerequisites
- A running Actian VectorAI instance (default: `localhost:50051`)
- An OpenAI API key for embeddings/LLM

### 1. Install dependencies

In [ ]:
%pip install -q llama-index llama-index-llms-openai llama-index-embeddings-openai llama-index-vector-stores-actian-vectorai

### 2. Configure credentials and imports

In [ ]:
import os
import uuid
from getpass import getpass

from llama_index.core import (
    Document,
    Settings,
    StorageContext,
    VectorStoreIndex,
)
from llama_index.core.vector_stores.types import (
    MetadataFilter,
    MetadataFilters,
    FilterOperator,
    VectorStoreQuery,
)
from llama_index.embeddings.openai import OpenAIEmbedding
from llama_index.llms.openai import OpenAI
from llama_index.vector_stores.actian_vectorai import ActianVectorAIVectorStore

if not os.getenv("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass("Enter OPENAI_API_KEY: ")

VECTORAI_SERVER_URL = os.getenv("VECTORAI_SERVER_URL", "localhost:50051")
COLLECTION_NAME = f"llama_index_actian_demo_{uuid.uuid4().hex[:8]}"

print(f"Using Actian endpoint: {VECTORAI_SERVER_URL}")
print(f"Using collection: {COLLECTION_NAME}")

### 3. Create sample documents and build an index

In [ ]:
documents = [
    Document(
        text="LlamaIndex helps orchestrate RAG pipelines and retrieval over unstructured data.",
        metadata={"topic": "rag", "year": 2023},
    ),
    Document(
        text="Actian VectorAI is a vector database used for semantic search and filtering.",
        metadata={"topic": "database", "year": 2024},
    ),
    Document(
        text="Metadata filters let you narrow retrieval results by fields like topic or year.",
        metadata={"topic": "rag", "year": 2022},
    ),
    Document(
        text="Production systems often combine vector retrieval with re-ranking and tool usage.",
        metadata={"topic": "architecture", "year": 2016},
    ),
]

Settings.embed_model = OpenAIEmbedding(model="text-embedding-3-small")
Settings.llm = OpenAI(model="gpt-4o-mini")

vector_store = ActianVectorAIVectorStore(
    url=VECTORAI_SERVER_URL,
    collection_name=COLLECTION_NAME,
    stores_text=True,
    clear_existing_collection=True,
)
vector_store.connect()

storage_context = StorageContext.from_defaults(vector_store=vector_store)
index = VectorStoreIndex.from_documents(
    documents, storage_context=storage_context
)

print(f"Indexed {len(documents)} documents into {COLLECTION_NAME}")

### 4. Run a semantic query

In [ ]:
query_engine = index.as_query_engine(similarity_top_k=2)
response = query_engine.query(
    "What does this demo say about vector databases?"
)
print(response)

### 5. Query with metadata filters

In [ ]:
query_embedding = Settings.embed_model.get_query_embedding(
    "Find content about retrieval architecture from 2016"
)

filters = MetadataFilters(
    filters=[
        MetadataFilter(
            key="topic", operator=FilterOperator.EQ, value="architecture"
        ),
        MetadataFilter(key="year", operator=FilterOperator.GTE, value=2016),
    ]
)

result = vector_store.query(
    VectorStoreQuery(
        query_embedding=query_embedding,
        similarity_top_k=3,
        filters=filters,
    )
)

for i, node in enumerate(result.nodes, start=1):
    print(f"Result {i}: {node.get_content()}")
    print(f"Metadata: {node.metadata}")
    print("-" * 80)

### 6. Cleanup (optional but recommended)

In [ ]:
# Remove demo data and close the client connection.
vector_store.clear()
vector_store.close()
print(f"Cleared and closed collection: {COLLECTION_NAME}")